# Fatal Force: deaths involving police in the United States

Day 100 assignment of the Udemy course *100 Days of Code: The Complete Python Pro Bootcamp*. The Washington Post has recorded every fatal shooting by a police officer in the line of duty since 1 January 2015. This notebook combines that database with US census data.

The topic is sensitive. The charts show counts and rates; they do not show causes. Read each chart with its caption.

The logic lives in the package `fatal_force` (`src/fatal_force/`); this notebook calls it and shows the result. Each step saves a screenshot in `docs/screenshots/`.

In [1]:
import polars as pl
from IPython.display import display

from fatal_force.infrastructure.cleaning import clean_all
from fatal_force.infrastructure.csv_reader import load_datasets
from fatal_force.infrastructure.exploration import summarise_frame
from fatal_force.presentation.screenshots import save_figure, summary_table_figure

pl.Config.set_tbl_rows(8)

polars.config.Config

## Load the data

The five course files are read from the git-ignored `data/` folder (see the README for how to obtain them).

In [2]:
raw = load_datasets()
raw_frames = {
    "Median household income": raw.hh_income,
    "Poverty rate": raw.pct_poverty,
    "High school completion": raw.pct_completed_hs,
    "Share of race by city": raw.share_race_city,
    "Deaths by police": raw.fatalities,
}

## Step 1: Preliminary data exploration

What is the shape of each frame, what are its columns, and are there missing values or duplicates?

In [3]:
raw_summaries = [summarise_frame(name, frame) for name, frame in raw_frames.items()]
for summary in raw_summaries:
    print(
        f"{summary.name}: {summary.rows} rows x {summary.columns} columns, "
        f"{summary.null_total} missing, {summary.duplicate_rows} duplicate rows"
    )
    print("   columns:", ", ".join(summary.column_names))
display(raw.fatalities.head())

Median household income: 29322 rows x 3 columns, 51 missing, 0 duplicate rows
   columns: Geographic Area, City, Median Income
Poverty rate: 29329 rows x 3 columns, 0 missing, 0 duplicate rows
   columns: Geographic Area, City, poverty_rate
High school completion: 29329 rows x 3 columns, 0 missing, 0 duplicate rows
   columns: Geographic Area, City, percent_completed_hs
Share of race by city: 29268 rows x 7 columns, 0 missing, 0 duplicate rows
   columns: Geographic area, City, share_white, share_black, share_native_american, share_asian, share_hispanic
Deaths by police: 2535 rows x 14 columns, 346 missing, 0 duplicate rows
   columns: id, name, date, manner_of_death, armed, age, gender, race, city, state, signs_of_mental_illness, threat_level, flee, body_camera


id,name,date,manner_of_death,armed,age,gender,race,city,state,signs_of_mental_illness,threat_level,flee,body_camera
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""3""","""Tim Elliot""","""02/01/15""","""shot""","""gun""","""53""","""M""","""A""","""Shelton""","""WA""","""TRUE""","""attack""","""Not fleeing""","""FALSE"""
"""4""","""Lewis Lee Lembke""","""02/01/15""","""shot""","""gun""","""47""","""M""","""W""","""Aloha""","""OR""","""FALSE""","""attack""","""Not fleeing""","""FALSE"""
"""5""","""John Paul Quintero""","""03/01/15""","""shot and Tasered""","""unarmed""","""23""","""M""","""H""","""Wichita""","""KS""","""FALSE""","""other""","""Not fleeing""","""FALSE"""
"""8""","""Matthew Hoffman""","""04/01/15""","""shot""","""toy weapon""","""32""","""M""","""W""","""San Francisco""","""CA""","""TRUE""","""attack""","""Not fleeing""","""FALSE"""
"""9""","""Michael Rodriguez""","""04/01/15""","""shot""","""nail gun""","""39""","""M""","""H""","""Evans""","""CO""","""FALSE""","""attack""","""Not fleeing""","""FALSE"""


In [4]:
figure = summary_table_figure("Step 1: raw data exploration", raw_summaries)
save_figure(figure, 1, "data-exploration")
figure

<Figure size 800x400 with 1 Axes>

## Step 2: Data cleaning

Census placeholders such as `(X)` and `-` become missing values, thousands separators and bounds such as `250,000+` are removed so the columns are numbers, the date column is parsed, and missing labels in the deaths data become `unknown`. Missing census values are kept as missing, not turned into zero, so they do not pull averages down.

In [5]:
clean = clean_all(raw)
clean_frames = {
    "Median household income": clean.hh_income,
    "Poverty rate": clean.pct_poverty,
    "High school completion": clean.pct_completed_hs,
    "Share of race by city": clean.share_race_city,
    "Deaths by police": clean.fatalities,
}
clean_summaries = [summarise_frame(name, frame) for name, frame in clean_frames.items()]
for before, after in zip(raw_summaries, clean_summaries, strict=True):
    print(f"{before.name}: missing {before.null_total} -> {after.null_total}")
display(clean.fatalities.head())

Median household income: missing 51 -> 1904
Poverty rate: missing 0 -> 201
High school completion: missing 0 -> 197
Share of race by city: missing 0 -> 100
Deaths by police: missing 346 -> 77


id,name,date,manner_of_death,armed,age,gender,race,city,state,signs_of_mental_illness,threat_level,flee,body_camera
i64,str,date,str,str,i64,str,str,str,str,bool,str,str,bool
3,"""Tim Elliot""",2015-01-02,"""shot""","""gun""",53,"""M""","""A""","""Shelton""","""WA""",true,"""attack""","""Not fleeing""",false
4,"""Lewis Lee Lembke""",2015-01-02,"""shot""","""gun""",47,"""M""","""W""","""Aloha""","""OR""",false,"""attack""","""Not fleeing""",false
5,"""John Paul Quintero""",2015-01-03,"""shot and Tasered""","""unarmed""",23,"""M""","""H""","""Wichita""","""KS""",false,"""other""","""Not fleeing""",false
8,"""Matthew Hoffman""",2015-01-04,"""shot""","""toy weapon""",32,"""M""","""W""","""San Francisco""","""CA""",true,"""attack""","""Not fleeing""",false
9,"""Michael Rodriguez""",2015-01-04,"""shot""","""nail gun""",39,"""M""","""H""","""Evans""","""CO""",false,"""attack""","""Not fleeing""",false


In [6]:
figure = summary_table_figure("Step 2: data after cleaning", clean_summaries)
save_figure(figure, 2, "data-cleaning")
figure

<Figure size 800x400 with 1 Axes>

## Step 3: Poverty rate in each US state

The poverty rate of a state is the mean of the poverty rates of its cities in the census file. Missing city values are ignored.

In [7]:
from fatal_force.infrastructure.aggregation import (
    correlation,
    high_school_rate_by_state,
    poverty_and_graduation_by_state,
    poverty_rate_by_state,
)
from fatal_force.presentation.charts import (
    dual_axis_line_chart,
    ranked_bar_chart,
    scatter_with_fit_chart,
)

poverty = poverty_rate_by_state(clean.pct_poverty)
print("Highest:", poverty.row(0), " Lowest:", poverty.row(-1))
figure = ranked_bar_chart(
    poverty,
    "state",
    "poverty_rate",
    "Poverty rate by US state",
    "Mean city poverty rate (%)",
)
save_figure(figure, 3, "poverty-rate")
figure

Highest: ('MS', 26.88425414364641)  Lowest: ('NJ', 8.19097605893186)


E:\repos\Tirsvad-Udemy-100-days-of-code\100-assignment-analyse-deaths-involving-police-in-the-united-states\src\fatal_force\presentation\charts.py:54: UserWarning: The figure layout has changed to tight
  figure.tight_layout(rect=(0, _NOTE_MARGIN, 1, 1))


<Figure size 900x1628 with 1 Axes>

## Step 4: High school graduation rate by US state

Share of people over 25 who completed high school, again as the mean of the cities of each state, lowest first.

In [8]:
graduation = high_school_rate_by_state(clean.pct_completed_hs)
print("Lowest:", graduation.row(0), " Highest:", graduation.row(-1))
figure = ranked_bar_chart(
    graduation,
    "state",
    "percent_completed_hs",
    "High school graduation rate by US state",
    "Mean city completion rate (%)",
)
save_figure(figure, 4, "graduation-rate")
figure

Lowest: ('TX', 75.69)  Highest: ('MA', 92.40408163265306)


<Figure size 900x1628 with 1 Axes>

## Step 5: Do poverty and graduation move together?

A line chart with two y-axes puts both rates next to each other, state by state.

In [9]:
combined = poverty_and_graduation_by_state(poverty, graduation)
figure = dual_axis_line_chart(
    combined,
    "state",
    "poverty_rate",
    "percent_completed_hs",
    "Poverty rate and high school graduation rate by state",
    "Poverty rate (%)",
    "High school graduation rate (%)",
)
save_figure(figure, 5, "poverty-and-graduation-lines")
figure

<Figure size 1200x600 with 2 Axes>

## Step 6: Scatter plot with a regression line

Each dot is a state. The line is the least-squares fit. A strong correlation shows that the two measures move together; it does not show that one causes the other.

In [10]:
print(
    "Pearson r:",
    round(correlation(combined, "poverty_rate", "percent_completed_hs"), 4),
)
figure = scatter_with_fit_chart(
    combined,
    "poverty_rate",
    "percent_completed_hs",
    "Poverty rate against high school graduation rate",
    "Poverty rate (%)",
    "High school graduation rate (%)",
    "state",
)
save_figure(figure, 6, "poverty-vs-graduation-scatter")
figure

Pearson r: -0.8617


<Figure size 800x600 with 1 Axes>

## Step 7: Racial makeup of each US state

The census file gives the share of five groups in every city. The share of a state is the plain mean of its cities (cities are not weighted by population, so large cities count as much as small ones). The census treats Hispanic as an ethnicity that overlaps the race groups, so the five shares of a state can add up to more than 100 %.

In [11]:
from fatal_force.constants import SHARE_COLUMNS, SHARE_LABELS
from fatal_force.infrastructure.aggregation import (
    deaths_by_race,
    racial_makeup_by_state,
)
from fatal_force.presentation.charts import donut_chart, stacked_bar_chart

makeup = racial_makeup_by_state(clean.share_race_city)
figure = stacked_bar_chart(
    makeup,
    "state",
    list(SHARE_COLUMNS),
    list(SHARE_LABELS),
    "Racial makeup of each US state",
    "Mean city share (%)",
)
save_figure(figure, 7, "racial-makeup-by-state")
figure

<Figure size 1400x600 with 1 Axes>

## Step 8: People killed, by race

The Washington Post records the race of each person. For 195 of the 2,535 records it is missing; these are shown as *Unknown*. The chart shows counts, not rates: it says nothing about how likely a person of a given race is to be killed, because the groups differ in size.

In [12]:
by_race = deaths_by_race(clean.fatalities)
print(by_race)
figure = donut_chart(by_race, "race", "deaths", "People killed by police, by race")
save_figure(figure, 8, "deaths-by-race")
figure

shape: (7, 3)
┌───────────┬─────────────────┬────────┐
│ race_code ┆ race            ┆ deaths │
│ ---       ┆ ---             ┆ ---    │
│ str       ┆ str             ┆ u32    │
╞═══════════╪═════════════════╪════════╡
│ W         ┆ White           ┆ 1201   │
│ B         ┆ Black           ┆ 618    │
│ H         ┆ Hispanic        ┆ 423    │
│ unknown   ┆ Unknown         ┆ 195    │
│ A         ┆ Asian           ┆ 39     │
│ N         ┆ Native American ┆ 31     │
│ O         ┆ Other           ┆ 28     │
└───────────┴─────────────────┴────────┘


<Figure size 800x600 with 1 Axes>

## Step 9: Deaths of men and women

Counts of people killed by gender, and how they died (*shot*, or *shot and Tasered*).

In [13]:
from fatal_force.infrastructure.victims import (
    ages_by_group,
    armed_counts,
    armed_shares,
    deaths_by_gender,
    deaths_by_manner,
    flee_counts,
    mental_illness_share,
    share_under_age,
    threat_level_counts,
)
from fatal_force.presentation.charts import bar_chart
from fatal_force.presentation.distributions import (
    box_plot_chart,
    histogram_with_kde_chart,
    kde_by_group_chart,
)

by_gender = deaths_by_gender(clean.fatalities)
print(by_gender)
print(deaths_by_manner(clean.fatalities))
figure = bar_chart(
    by_gender, "value", "count", "People killed by police, by gender", "Deaths"
)
save_figure(figure, 9, "deaths-by-gender")
figure

shape: (2, 2)
┌───────┬───────┐
│ value ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ M     ┆ 2428  │
│ F     ┆ 107   │
└───────┴───────┘
shape: (4, 3)
┌────────┬──────────────────┬────────┐
│ gender ┆ manner_of_death  ┆ deaths │
│ ---    ┆ ---              ┆ ---    │
│ str    ┆ str              ┆ u32    │
╞════════╪══════════════════╪════════╡
│ F      ┆ shot             ┆ 102    │
│ F      ┆ shot and Tasered ┆ 5      │
│ M      ┆ shot             ┆ 2261   │
│ M      ┆ shot and Tasered ┆ 167    │
└────────┴──────────────────┴────────┘


<Figure size 1000x600 with 1 Axes>

## Step 10: How old were the people killed?

In [14]:
ages = ages_by_group(clean.fatalities, "gender")
all_ages = [age for group in ages.values() for age in group]
print("Known ages:", len(all_ages))
print("Under 25:", round(share_under_age(clean.fatalities, 25), 1), "%")
figure = histogram_with_kde_chart(
    all_ages, "Age of people killed by police", "Age (years)", 30
)
save_figure(figure, 10, "age-distribution")
figure

Known ages: 2458
Under 25: 18.3 %


<Figure size 1000x600 with 1 Axes>

## Step 11: Age and manner of death

A box plot of age for each manner of death. The *shot and Tasered* group is much smaller, so compare its box with care.

In [15]:
by_manner = ages_by_group(clean.fatalities, "manner_of_death")
print({name: len(values) for name, values in by_manner.items()})
figure = box_plot_chart(by_manner, "Age by manner of death", "Age (years)")
save_figure(figure, 11, "age-by-manner-of-death")
figure

{'shot': 2290, 'shot and Tasered': 168}


E:\repos\Tirsvad-Udemy-100-days-of-code\100-assignment-analyse-deaths-involving-police-in-the-united-states\src\fatal_force\presentation\charts.py:54: UserWarning: The figure layout has changed to tight
  figure.tight_layout(rect=(0, _NOTE_MARGIN, 1, 1))


<Figure size 1000x600 with 1 Axes>

## Step 12: Were the people armed?

*Armed* here means every weapon label except *unarmed*, including *undetermined*; *unknown* is counted separately.

In [16]:
shares = armed_shares(clean.fatalities)
print(
    f"Armed {shares.armed_percent:.1f} %, unarmed {shares.unarmed_percent:.1f} %, "
    f"unknown {shares.unknown_percent:.1f} %"
)
weapons = armed_counts(clean.fatalities).head(10)
figure = bar_chart(
    weapons,
    "value",
    "count",
    "Weapon carried by the person killed (top 10)",
    "Deaths",
    horizontal=True,
)
save_figure(figure, 12, "armed-and-unarmed")
figure

Armed 92.9 %, unarmed 6.7 %, unknown 0.4 %


<Figure size 1000x600 with 1 Axes>

## Step 13: Age distribution by race

One density line per group. Groups with few records give rough lines; the legend shows each group's *n*.

In [17]:
by_race = ages_by_group(clean.fatalities, "race")
figure = kde_by_group_chart(by_race, "Age distribution by race", "Age (years)")
save_figure(figure, 13, "age-by-race")
figure

<Figure size 1000x600 with 1 Axes>

## Step 14: Mental illness and the threat level

In [18]:
mental = mental_illness_share(clean.fatalities)
print(f"Signs of mental illness recorded: {mental:.1f} %")
mental_frame = pl.DataFrame(
    {
        "value": ["Signs of mental illness", "No signs recorded"],
        "count": [round(mental, 1), round(100 - mental, 1)],
    }
)
figure = bar_chart(
    mental_frame,
    "value",
    "count",
    "Signs of mental illness (share of records, %)",
    "Share of records (%)",
)
save_figure(figure, 14, "mental-illness")
figure

Signs of mental illness recorded: 25.0 %


<Figure size 1000x600 with 1 Axes>

## Step 15: Threat level and fleeing

In [19]:
print(threat_level_counts(clean.fatalities))
figure = bar_chart(
    flee_counts(clean.fatalities),
    "value",
    "count",
    "Fleeing status of the person killed",
    "Deaths",
)
save_figure(figure, 15, "threat-and-fleeing")
figure

shape: (3, 2)
┌──────────────┬───────┐
│ value        ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ attack       ┆ 1611  │
│ other        ┆ 766   │
│ undetermined ┆ 158   │
└──────────────┴───────┘


<Figure size 1000x600 with 1 Axes>

## Step 16: In which cities do the most police killings take place?

The ten cities with the most killings in the database. These are counts, not rates: large cities have more people.

In [20]:
from fatal_force.infrastructure.geography import (
    deaths_by_state,
    ratio_by_city_and_race,
    top_cities,
    victim_and_population_shares,
)
from fatal_force.infrastructure.timeline import (
    deaths_per_month,
    deaths_per_year,
    rolling_mean,
    trend,
)
from fatal_force.presentation.charts import grouped_bar_chart, line_chart
from fatal_force.presentation.maps import choropleth_figure, save_plotly_figure

cities = top_cities(clean.fatalities, 10)
labelled = cities.with_columns((pl.col("city") + ", " + pl.col("state")).alias("label"))
print(labelled)
figure = bar_chart(
    labelled,
    "label",
    "deaths",
    "Ten cities with the most police killings",
    "Deaths",
    horizontal=True,
)
save_figure(figure, 16, "top-cities")
figure

shape: (10, 4)
┌─────────────┬───────┬────────┬─────────────────┐
│ city        ┆ state ┆ deaths ┆ label           │
│ ---         ┆ ---   ┆ ---    ┆ ---             │
│ str         ┆ str   ┆ u32    ┆ str             │
╞═════════════╪═══════╪════════╪═════════════════╡
│ Los Angeles ┆ CA    ┆ 39     ┆ Los Angeles, CA │
│ Phoenix     ┆ AZ    ┆ 31     ┆ Phoenix, AZ     │
│ Houston     ┆ TX    ┆ 26     ┆ Houston, TX     │
│ Chicago     ┆ IL    ┆ 25     ┆ Chicago, IL     │
│ …           ┆ …     ┆ …      ┆ …               │
│ Columbus    ┆ OH    ┆ 17     ┆ Columbus, OH    │
│ Miami       ┆ FL    ┆ 17     ┆ Miami, FL       │
│ Austin      ┆ TX    ┆ 16     ┆ Austin, TX      │
│ St. Louis   ┆ MO    ┆ 15     ┆ St. Louis, MO   │
└─────────────┴───────┴────────┴─────────────────┘


<Figure size 1000x600 with 1 Axes>

## Step 17: Rate of death by race in the top ten cities

For each city we compare the share of the people killed who belong to a race with the census share of that race in the city. A ratio of 1 means the shares match; above 1 means the group is a larger share of the people killed than of the city. Each city has only 15 to 39 killings, so single ratios are rough, and a group with no killings shows 0. The census Hispanic share overlaps the race shares.

In [21]:
shares = victim_and_population_shares(clean.fatalities, clean.share_race_city, cities)
ratios = ratio_by_city_and_race(shares, ["W", "B", "H"])
print(ratios)
figure = grouped_bar_chart(
    ratios,
    "city_label",
    ["W", "B", "H"],
    ["White", "Black", "Hispanic"],
    "Share of people killed relative to census share, top ten cities",
    "Ratio (1 = equal shares)",
)
save_figure(figure, 17, "rate-by-race-top-cities")
figure

shape: (10, 4)
┌─────────────────┬──────────┬──────────┬──────────┐
│ city_label      ┆ W        ┆ B        ┆ H        │
│ ---             ┆ ---      ┆ ---      ┆ ---      │
│ str             ┆ f64      ┆ f64      ┆ f64      │
╞═════════════════╪══════════╪══════════╪══════════╡
│ Los Angeles, CA ┆ 0.334672 ┆ 2.893519 ┆ 1.088202 │
│ Phoenix, AZ     ┆ 0.650336 ┆ 1.098901 ┆ 0.962885 │
│ Houston, TX     ┆ 0.237624 ┆ 2.531646 ┆ 0.547945 │
│ Chicago, IL     ┆ 0.185185 ┆ 2.659574 ┆ 0.144175 │
│ …               ┆ …        ┆ …        ┆ …        │
│ Columbus, OH    ┆ 0.50813  ┆ 2.232143 ┆ 0.0      │
│ Miami, FL       ┆ 0.172176 ┆ 2.604167 ┆ 0.535714 │
│ Austin, TX      ┆ 1.006589 ┆ 2.314815 ┆ 0.356125 │
│ St. Louis, MO   ┆ 0.488122 ┆ 1.59698  ┆ 0.0      │
└─────────────────┴──────────┴──────────┴──────────┘


E:\repos\Tirsvad-Udemy-100-days-of-code\100-assignment-analyse-deaths-involving-police-in-the-united-states\src\fatal_force\presentation\charts.py:54: UserWarning: The figure layout has changed to tight
  figure.tight_layout(rect=(0, _NOTE_MARGIN, 1, 1))


<Figure size 1400x600 with 1 Axes>

## Step 18: Choropleth map of police killings by state

Counts per state. California, Texas and Florida lead, which follows their size; compare the map with the poverty chart of step 3 before drawing a conclusion.

In [22]:
by_state = deaths_by_state(clean.fatalities)
print(by_state.head(10))
map_figure = choropleth_figure(
    by_state, "state", "deaths", "Police killings by US state", "Deaths"
)
save_plotly_figure(map_figure, 18, "deaths-by-state-map")
map_figure

shape: (10, 2)
┌───────┬────────┐
│ state ┆ deaths │
│ ---   ┆ ---    │
│ str   ┆ u32    │
╞═══════╪════════╡
│ CA    ┆ 424    │
│ TX    ┆ 225    │
│ FL    ┆ 154    │
│ AZ    ┆ 118    │
│ …     ┆ …      │
│ CO    ┆ 74     │
│ GA    ┆ 70     │
│ NC    ┆ 69     │
│ MO    ┆ 64     │
└───────┴────────┘


## Step 19: Number of police killings over time

Deaths per month, with a three-month rolling mean. The data covers January 2015 to July 2017, and the straight-line trend over the months is almost flat.

In [23]:
monthly = rolling_mean(deaths_per_month(clean.fatalities), "deaths", 3)
fit = trend(monthly)
slope = f"{fit.slope:+.3f}"
print(f"{monthly.height} months, trend {slope} deaths per month, r = {fit.r_value:.2f}")
print(deaths_per_year(clean.fatalities))
figure = line_chart(
    monthly,
    "month",
    "deaths",
    "Police killings per month",
    "Deaths per month",
    smooth_column="deaths_rolling_mean",
)
save_figure(figure, 19, "deaths-over-time")
figure

31 months, trend -0.038 deaths per month, r = -0.04
shape: (3, 2)
┌──────┬────────┐
│ year ┆ deaths │
│ ---  ┆ ---    │
│ i32  ┆ u32    │
╞══════╪════════╡
│ 2015 ┆ 991    │
│ 2016 ┆ 963    │
│ 2017 ┆ 581    │
└──────┴────────┘


E:\repos\Tirsvad-Udemy-100-days-of-code\100-assignment-analyse-deaths-involving-police-in-the-united-states\src\fatal_force\presentation\charts.py:54: UserWarning: The figure layout has changed to tight
  figure.tight_layout(rect=(0, _NOTE_MARGIN, 1, 1))


<Figure size 1200x600 with 1 Axes>